# ArcLog 練投紀錄員：訓練籃球偵測模型

這份筆記本在 **Google Colab** 上從頭到尾執行一次，就會得到一個網站可以用的籃球偵測模型（ONNX 檔）。
流程是：**安裝套件 → 準備資料 → 訓練 → 看訓練結果 → 匯出 ONNX → 評估 → 下載結果**。

每一格程式的上面都有一段說明，講這一格在做什麼。照順序一格一格按 ▶ 執行就可以；
或是用選單的「執行階段 → 全部執行」。

## 先決定要跑哪一個階段

| 階段 | 用什麼資料 | 需要先準備什麼 |
|---|---|---|
| `B` 公開資料訓練 | Roboflow Universe 上 CC BY 4.0 授權的籃球資料集 | 一組免費的 Roboflow API 金鑰（申請方式見 `docs/NEXT_STEPS.md`） |
| `C` 自有資料微調 | 自己拍的練投影片＋在網站上標記後匯出的「標註包」 | 把影片和標註包放到自己的 Google Drive |
| `synthetic` 只試流程 | 程式畫出來的合成畫面 | 什麼都不用。只能確認流程會動，**訓練出來的模型不能用在真實影片上** |

模型只學一件事：在畫面裡找出**籃球**。籃框的位置由使用者在網站上點兩下校正，不需要訓練。

## 開始之前

1. 選單「執行階段 → 變更執行階段類型」，硬體加速器選 **T4 GPU**。沒有 GPU 也能跑，但會慢幾十倍。
2. 實拍影片裡有人。只有在球員和教練同意之後，才可以把影片放到自己的 Google Drive 來訓練；
   影片和抽出來的畫面**不可以**放進公開的 GitHub repo（說明見 `docs/DATA.md`）。

## 1. 設定

這一格只是填設定，不會真的做事。最重要的是第一行 `STAGE`：填 `"B"`、`"C"` 或 `"synthetic"`。

其他設定平常不用改：

- `EPOCHS`：最多訓練幾輪（把全部訓練資料看過一遍叫一輪）。填 `None` 會用預設值（B：80、C：50、synthetic：3），
  而且驗證分數連續很多輪沒有進步時會自動提早停。
- `PUBLIC_DATASETS`：階段 B 要下載的公開資料集。出處、授權、為什麼選這幾個，寫在 `docs/DATA.md`。
- `DATA_DIR`：階段 C 放資料的 Google Drive 資料夾。裡面要有一個 `videos` 資料夾，放影片和同名的標註包，例如
  `練投_1108.mov` 和 `練投_1108.arclog.json`。

In [ ]:
STAGE = "B"          # "B"＝公開資料集；"C"＝自己的影片；"synthetic"＝只用合成資料試流程
EPOCHS = None        # None＝用預設值

REPO_URL = "https://github.com/ryanwu566/arclog.git"
BRANCH = "main"      # PR 還沒合併時，改成 "feat/arclog-mvp"

# 階段 B：要下載的公開資料集（都是 CC BY 4.0 授權；細節見 docs/DATA.md）
# aspect："keep"＝影像維持原樣；"16:9"＝這個資料集的影像被壓成正方形，還原成 16:9 讓球變回圓的
PUBLIC_DATASETS = [
    {"workspace": "hooper-ibdsr", "project": "basketball-detection-v6", "version": 1, "aspect": "keep"},
    {"workspace": "xinghans-workspace", "project": "shot-wi85e", "version": 3, "aspect": "16:9"},
]

# 階段 C：Google Drive 上放資料的資料夾（裡面要有 videos/）
DATA_DIR = "/content/drive/MyDrive/arclog_data"

# ── 以下不用改 ──
import os
STAGE = os.environ.get("ARCLOG_STAGE", STAGE)                    # 自動測試時用環境變數覆蓋，平常不用管
EPOCHS = int(os.environ["ARCLOG_EPOCHS"]) if os.environ.get("ARCLOG_EPOCHS") else EPOCHS
DATA_DIR = os.environ.get("ARCLOG_DATA_DIR", DATA_DIR)
assert STAGE in ("B", "C", "synthetic"), 'STAGE 只能是 "B"、"C" 或 "synthetic"'
MODEL_ID = {"B": "arclog-ball-b-public", "C": "arclog-ball-c-own", "synthetic": "arclog-ball-synth"}[STAGE]
RUN_NAME = {"B": "stage_b", "C": "stage_c", "synthetic": "synthetic"}[STAGE]
print("這次要跑的階段：", STAGE, "；模型代號：", MODEL_ID)

## 2. 檢查有沒有 GPU

訓練神經網路要做非常大量的矩陣運算，顯示卡（GPU）做這件事比 CPU 快幾十倍。
這一格用 `nvidia-smi` 這個指令看這台機器有沒有 NVIDIA 的顯示卡。

如果顯示「沒有 GPU」：到選單「執行階段 → 變更執行階段類型」選 T4 GPU，然後從頭再執行一次。

In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
else:
    print("這台機器沒有 NVIDIA GPU：可以繼續，但訓練會很慢（只適合 synthetic 試流程）。")

## 3. 取得程式

把 ArcLog 的程式從 GitHub 複製（clone）到 Colab 的機器上。訓練、評估、匯出用的腳本都在裡面的 `training/` 資料夾。

如果不是在 Colab、而是在自己電腦上打開這份筆記本，這一格會直接使用電腦上現有的程式，不會重新下載。

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content/arclog")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "training" / "train.py").exists())
assert (ROOT / "training" / "train.py").exists(), '這個分支裡找不到 training/：請把第 1 格的 BRANCH 改成 "feat/arclog-mvp" 再從頭執行'
os.chdir(ROOT)
print("程式位置：", ROOT)

## 4. 安裝套件

安裝需要的 Python 套件（只有在 Colab 才會安裝；自己的電腦請照 README 先裝好）：

- `ultralytics`：YOLO 模型的訓練程式。這個專案用的是 YOLO26 的 Nano 尺寸（`yolo26n`），是最小、最快的一種，適合在手機瀏覽器裡跑。
- `onnx`、`onnxruntime`：模型匯出成 ONNX 格式、以及用和網站相同的方式執行模型。
- `av`：讀影片（內含 FFmpeg）。
- `roboflow`：下載公開資料集（只有階段 B 用到）。

這一格大約要一兩分鐘。

In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-r", "training/requirements.txt"], check=True)

import torch
import ultralytics

print("ultralytics", ultralytics.__version__, "｜torch", torch.__version__, "｜可以用 GPU：", torch.cuda.is_available())

## 5. 一個小工具：執行腳本並顯示它印出來的訊息

接下來每個步驟都是呼叫 `training/` 裡的一支 Python 腳本。這一格定義一個小函式 `run(...)`，
作用是「執行一支腳本，把它印出來的訊息即時顯示在這裡；如果腳本回報失敗就停下來」。

這樣筆記本和自己電腦上的指令用的是**同一份程式**：在電腦上打
`python training/train.py --stage B --data ...` 和在這裡寫 `run("training/train.py", "--stage", "B", "--data", ...)` 是同一件事。

In [ ]:
def run(*args):
    """執行 repo 裡的一支腳本，把它印出來的訊息即時顯示在這裡。"""
    cmd = [sys.executable, *[str(a) for a in args]]
    env = dict(os.environ, PYTHONUTF8="1", PYTHONIOENCODING="utf-8")
    proc = subprocess.Popen(cmd, cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError("這一步失敗了，請看上面的訊息：" + " ".join(cmd[1:]))


def show(path, width=1100):
    """把一張圖顯示在筆記本裡。"""
    from IPython.display import Image, display

    if Path(path).exists():
        display(Image(filename=str(path), width=width))
    else:
        print("找不到圖：", path)

## 6. 準備資料

三個階段準備資料的方式不一樣，下面三格**只有符合 `STAGE` 的那一格會真的執行**，其他兩格會直接跳過。
不管哪一個階段，最後都會得到一個資料夾，裡面是「畫面＋每張畫面上籃球的框」，以及一個 `data.yaml` 告訴訓練程式資料在哪裡。

### 6-B. 階段 B：下載公開資料集，只留下「籃球」這個類別

1. 執行後會出現一個輸入欄位，請貼上 Roboflow 的 **Private API Key**。
   金鑰只存在這次執行的記憶體裡，用完就刪掉；**不會顯示在畫面上，也不會存進筆記本或 repo**。請不要把金鑰直接寫在程式裡。
2. 下載 `PUBLIC_DATASETS` 列的資料集。
3. `prepare_public.py` 把資料整理成只有籃球一個類別：公開資料集通常同時標了球、球員、籃框，我們只要球，其他的框全部丟掉。

注意：公開資料集的畫面和我們的拍法（罰球線正側面、固定腳架）不完全一樣，所以這個階段的模型在我們自己影片上的表現，
要等有實拍影片之後才量得出來。

In [ ]:
if STAGE == "B":
    from getpass import getpass

    from roboflow import Roboflow

    api_key = getpass("貼上 Roboflow 的 Private API Key，然後按 Enter：").strip()
    assert api_key, "沒有輸入金鑰"
    rf = Roboflow(api_key=api_key)
    prepare_args = []
    for d in PUBLIC_DATASETS:
        dest = ROOT / "datasets" / f"{d['project']}-{d['version']}"
        if not (dest / "data.yaml").exists():
            version = rf.workspace(d["workspace"]).project(d["project"]).version(d["version"])
            last_error = None
            for fmt in ("yolo26", "yolov11", "yolov8"):  # 這幾種格式的標註檔內容相同，哪一種下載得到就用哪一種
                try:
                    version.download(fmt, location=str(dest))
                    last_error = None
                    break
                except Exception as err:  # noqa: BLE001
                    last_error = err
            if last_error is not None:
                raise last_error
        prepare_args += ["--src", dest, "--aspect", d.get("aspect", "keep")]
    del api_key, rf  # 金鑰用完就丟掉

    DATASET = ROOT / "datasets" / "public_ball"
    run("training/prepare_public.py", *prepare_args, "--out", DATASET, "--clean")
    show(DATASET / "preview_check.jpg")
else:
    print("不是階段 B，跳過這一格。")

### 6-C. 階段 C：用自己的影片和標註包

1. 掛載 Google Drive（會跳出視窗請你同意，這樣 Colab 才讀得到 Drive 裡的檔案）。
2. `build_dataset.py` 讀 `DATA_DIR/videos` 裡的每一支影片和它的標註包，把有標記的畫面抽出來，
   並且用**時間戳**確認「網站上看到的第幾幀」和「這裡解出來的第幾幀」是同一個畫面。
3. 資料以**影片**為單位分成訓練、驗證、測試三組：同一支影片的畫面只會在其中一組。
   如果同一支影片的畫面同時出現在訓練和測試，模型等於偷看過答案，測出來的分數會假性偏高。
4. `check_overlay.py` 隨機抽一些畫面，把框畫上去拼成一張圖。**請用眼睛看一下：綠色的框有沒有套在球上？**
   如果框都落在球的前面或後面，代表畫面沒有對齊，不要拿這份資料去訓練。

In [ ]:
if STAGE == "C":
    if IN_COLAB and DATA_DIR.startswith("/content/drive"):
        from google.colab import drive

        drive.mount("/content/drive")
    PAIRS = Path(DATA_DIR) / "videos"
    assert PAIRS.is_dir(), f"找不到 {PAIRS}：請把影片和標註包放進這個資料夾"
    DATASET = (Path("/content") if IN_COLAB else ROOT / "data") / "dataset_own"
    run("training/build_dataset.py", "--pairs", PAIRS, "--out", DATASET, "--clean")
    run("training/check_overlay.py", "--dataset", DATASET)
    show(DATASET / "overlay_check.jpg")
    show(DATASET / "overlay_negatives.jpg")
else:
    print("不是階段 C，跳過這一格。")

### 6-S. synthetic：用程式畫出來的合成畫面試流程

沒有金鑰也沒有影片時，可以用這個模式確認整條流程會動：

1. `make_synth.py` 用網站「示範模式」的同一份繪圖程式，畫出幾支合成的練投影片，並產生對應的標註包和真實答案。
   （繪圖程式是 JavaScript 寫的，所以這一步需要 Node.js 20 以上；Colab 通常已經內建。）
2. 之後和階段 C 一樣：`build_dataset.py` 抽畫面、`check_overlay.py` 疊圖檢查。
   合成資料知道每一幀球真正的位置，所以疊圖檢查還會直接算出「標註和真實位置差幾個像素」。

**合成畫面訓練出來的模型只認得合成的球，不代表在真實影片上的準確度。**

In [ ]:
if STAGE == "synthetic":
    assert shutil.which("node"), "找不到 Node.js：請先安裝 Node.js 20 以上（https://nodejs.org），或改跑階段 B／C"
    if not (ROOT / "node_modules").exists():
        subprocess.run("npm ci --no-audit --no-fund", cwd=ROOT, shell=True, check=True)
    SYNTH = ROOT / "data" / "nb_synth"
    DATASET = ROOT / "data" / "nb_dataset_synth"
    run("training/make_synth.py", "--out", SYNTH, "--videos", "3", "--shots", "5", "--small")
    run("training/build_dataset.py", "--pairs", SYNTH, "--out", DATASET, "--clean")
    run("training/check_overlay.py", "--dataset", DATASET, "--truth", SYNTH, "--max-error", "3")
    show(DATASET / "overlay_check.jpg")
else:
    print("不是 synthetic，跳過這一格。")

## 7. 訓練

`train.py` 做的事：

- 從一個**已經訓練過**的模型開始（階段 B 用官方在 COCO 資料集上訓練好的 `yolo26n.pt`；階段 C 用階段 B 的結果），
  再用我們的資料繼續訓練。這叫「微調」，比從零開始需要的資料少很多。
- 每一輪結束時，用**驗證組**（訓練時沒看過的資料）算一次分數，並把分數最好的那一版存成 `best.pt`。
- 驗證分數連續很多輪沒有進步就提早停，避免模型只是把訓練資料背起來（過度擬合）。

訓練過程會一直印出數字。`box_loss` 之類的 loss 越小代表模型在訓練資料上錯得越少；
`mAP50` 是偵測模型常用的綜合分數，越接近 1 越好。

階段 B 在 T4 GPU 上大約要一到兩小時（依資料量而定）；階段 C 和 synthetic 會快很多。

階段 C 會從 `DATA_DIR/stage_b_best.pt` 接著訓練（階段 B 做完後下載的結果裡有這個檔，請放到 Drive 的 `arclog_data` 資料夾）。
找不到這個檔的話，會改從官方的預訓練模型開始。

In [ ]:
train_args = ["--stage", STAGE, "--data", DATASET / "data.yaml"]
if EPOCHS:
    train_args += ["--epochs", EPOCHS]
if STAGE == "C":
    stage_b = Path(DATA_DIR) / "stage_b_best.pt"
    if stage_b.exists():
        train_args += ["--weights", stage_b]
    else:
        print(f"找不到 {stage_b}：改從官方預訓練模型開始訓練。")
if STAGE == "synthetic" and not torch.cuda.is_available():
    train_args += ["--batch", "8", "--workers", "2"]  # 沒有 GPU 時用小一點的設定，只求跑得完
run("training/train.py", *train_args)
RUN_DIR = ROOT / "runs" / "arclog" / RUN_NAME
BEST = RUN_DIR / "weights" / "best.pt"
assert BEST.exists(), "訓練沒有產生 best.pt"

## 8. 看訓練結果

訓練程式會畫出一張曲線圖：橫軸是第幾輪，縱軸是各種 loss 和分數。

怎麼看：

- loss 的曲線應該**往下走**然後變平。
- `metrics/recall`（有球的畫面裡找到球的比例）和 `metrics/precision`（框出來的東西有多少真的是球）應該**往上走**。
- 如果訓練的 loss 一直下降、但驗證的 loss 反而上升，代表開始過度擬合了。

這些是**驗證組**上的分數，只用來挑模型。模型實際好不好用，要看第 10 步在**測試影片**上的結果。

In [ ]:
show(RUN_DIR / "results.png")
print((RUN_DIR / "arclog_train.json").read_text(encoding="utf-8"))

## 9. 匯出 ONNX

網站是在瀏覽器裡執行模型的，瀏覽器不能直接讀 PyTorch 的 `best.pt`，所以要轉成通用的 **ONNX** 格式。

`export_onnx.py` 會：

1. 把 `best.pt` 匯出成 `models/<模型代號>.onnx`，輸入大小固定為寬 640、高 384（配合橫向 16:9 的影片）。
2. 用 `onnxruntime` 重新載入一次，確認檔案真的能用。
3. 把這個模型登記到 `models/manifest.json`（網站的模型選單就是讀這個檔），並複製一份到 `web/models/`。
   新模型只是「多一個選項」，網站預設用哪一個不會變；看過第 10 步的評估結果，確定比較好再改預設（做法見 `docs/NEXT_STEPS.md`）。

synthetic 模式的模型只是試流程用的，會放到 `data/nb_models_synth/`，不會登記到網站的模型清單裡。

In [ ]:
export_args = ["--weights", BEST, "--id", MODEL_ID, "--stage", STAGE]
if STAGE == "B":
    export_args += ["--data", "公開資料集：" + "、".join(f"{d['workspace']}/{d['project']} v{d['version']}" for d in PUBLIC_DATASETS) + "（CC BY 4.0）"]
if STAGE == "C":
    export_args += ["--data", "自有練投影片（未公開）"]
MODELS_DIR = ROOT / "models"
if STAGE == "synthetic":
    MODELS_DIR = ROOT / "data" / "nb_models_synth"
    export_args += ["--out-dir", MODELS_DIR, "--note", "合成資料訓練，只用來驗證流程，不代表真實準確度。"]
run("training/export_onnx.py", *export_args)
ONNX = MODELS_DIR / f"{MODEL_ID}.onnx"
print("ONNX 檔：", ONNX, f"（{ONNX.stat().st_size / 1e6:.1f} MB）")

## 10. 評估：在測試影片上跑完整的自動流程

前面的 `mAP50` 只說明「框畫得準不準」。但我們真正在意的是**最後算出來的入框角度準不準**，
所以 `evaluate.py` 是把模型放進和網站相同的自動流程（偵測 → 串成軌跡 → 擬合 → 算角度），再和人工標記的結果比較：

- **飛行球 recall／precision**：球在飛的那些畫面裡，模型有沒有找到球、框出來的是不是那顆球。
- **入框角度 MAE**：自動算出的角度和人工標記平均差幾度。
- **進球判定正確率**、**漏掉的球**、**多抓的球**。

所有模型用**同一批測試影片**比較，這樣才公平。測試影片是訓練時完全沒用過的影片。

- 階段 C：用 `build_dataset.py` 分出來的測試組，結果會寫進 `docs/EVAL.md`。
- 階段 B：如果 Drive 的 `DATA_DIR/videos` 已經有實拍影片和標註包，就用它們評估；沒有的話表格會寫「待實測」。
- synthetic：用合成影片評估，並和「真實答案」比較。這些數字**不代表真實準確度**。

In [ ]:
if STAGE == "synthetic":
    run("training/evaluate.py", "--dataset", DATASET, "--truth", SYNTH, "--section", "synthetic",
        "--models", f"arclog-ball-a-coco,{ONNX}")
elif STAGE == "C":
    run("training/evaluate.py", "--dataset", DATASET, "--write-doc")
else:
    pairs = Path(DATA_DIR) / "videos"
    if IN_COLAB and DATA_DIR.startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
        print("（沒有掛載 Google Drive，所以沒有實拍的測試影片。）")
    if pairs.is_dir() and any(pairs.glob("*.arclog.json")):
        run("training/evaluate.py", "--pairs", pairs, "--write-doc")
    else:
        run("training/evaluate.py")  # 沒有測試影片：表格會寫「待實測」

## 11. 下載結果

把這次的成果打包成一個 zip 下載：

| zip 裡的檔案 | 用途 |
|---|---|
| `models/<模型代號>.onnx` | 網站和 `tools/analyze.py` 用的模型 |
| `models/manifest.json` | 模型清單（已經登記好新模型） |
| `docs/EVAL.md` | 評估結果（有跑評估才會更新） |
| `weights/<階段>_best.pt` | PyTorch 權重。階段 B 的這個檔之後要給階段 C 接著訓練用，請留著 |
| `results.png`、`arclog_train.json` | 訓練曲線和這次訓練的設定 |

下載後怎麼放回專案、怎麼讓網站用新模型，請看 `docs/NEXT_STEPS.md`。簡單說：
把 zip 解壓縮到專案資料夾（覆蓋 `models/` 和 `docs/` 裡的同名檔案），執行 `python tools/sync_models.py`，再 commit 和 push。

注意：`.pt` 權重檔不要放進 repo（已經被 `.gitignore` 擋掉）；只提交 `.onnx`。

In [ ]:
import zipfile

OUT_ZIP = ROOT / "data" / f"arclog_result_{RUN_NAME}.zip"
OUT_ZIP.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX, f"models/{ONNX.name}")
    if STAGE != "synthetic":
        z.write(ROOT / "models" / "manifest.json", "models/manifest.json")
        z.write(ROOT / "docs" / "EVAL.md", "docs/EVAL.md")
    z.write(BEST, f"weights/{RUN_NAME}_best.pt")
    for name in ("results.png", "arclog_train.json"):
        if (RUN_DIR / name).exists():
            z.write(RUN_DIR / name, name)
print("已打包：", OUT_ZIP, f"（{OUT_ZIP.stat().st_size / 1e6:.1f} MB）")
if IN_COLAB:
    from google.colab import files

    files.download(str(OUT_ZIP))

## 做完之後

- **階段 B 做完**：把 zip 裡的 `weights/stage_b_best.pt` 放到 Google Drive 的 `arclog_data` 資料夾，之後階段 C 會用到。
- **想先試試新模型**：不用重新部署網站。打開網站 → 自動偵測 →「進階」→「載入自己訓練的 ONNX 模型」，選剛下載的 `.onnx` 就可以試。
- **確定要用**：照 `docs/NEXT_STEPS.md` 把模型放回專案並 push，Vercel 會自動重新部署。

授權提醒：模型是用 Ultralytics YOLO 訓練的，所以依 AGPL-3.0 釋出；階段 B 用到的公開資料集是 CC BY 4.0，
發表成果時要標示出處（清單在 `docs/DATA.md`）。